# A fairness audit of the attrition model, from scratch

The [main notebook](HRM_Employee%20Attrition.ipynb) in this repository predicts which
employees leave. Its winning model is a logistic regression over 59 encoded features, and
among those features are `Gender_Male`, `MaritalStatus_Married`, `MaritalStatus_Single`,
two `Department` dummies and two `Generation` dummies that are age wearing a different
label. The notebook fits that model, scores it, ranks employees by it, and never asks what
any of it would mean if a company ran it on Monday.

This notebook asks. It is a walkthrough rather than a report: every metric is computed in
front of you from a confusion matrix, with numpy and pandas and nothing else. No
`fairlearn`, no `aif360`. The arithmetic is small, and it is the teaching content. By the
end you should be able to run this audit on your own model without installing anything you
do not already have.

**Who this is for.** A data scientist who can fit a classifier and has never run a fairness
audit. You need pandas, numpy and scikit-learn. You need no legal background: the one rule
here that has a citation is quoted where it is used.

**The data is fictional, and that matters more than anything else in this notebook.** IBM
generated these 1,470 employees for a Watson Analytics demo. Nobody in the file resigned,
nobody was passed over for anything, and no group in it was disadvantaged by a real
employer. Every result below is therefore a measurement of a *method*, performed on
invented people. Read each one as "this is how you would find such a thing", never as "this
is what is happening to employees over 40". I will repeat this at the points where it is
easiest to forget.

The plan:

1. **Rebuild the model under audit**, exactly as the main notebook selects it, so the
   numbers here and the numbers there are the same numbers.
2. **Part 1, the audit.** Representation, selection rates, the four-fifths rule, equal
   opportunity, equalised odds, calibration, the confidence intervals that decide which of
   those differences you are allowed to believe, and a test of whether dropping a protected
   column would help.
3. **Part 2, the business impact.** What the confusion matrix costs, why 0.5 is a business
   decision rather than a technical default, and how a fairness gap turns into money.
4. **One summary image** and a checklist a reviewer can hand a client.

In [1]:
# Core python library
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 120)

# The same plotting stack, and the same calls, as the main notebook: every chart
# below is a plotly figure rendered with py.iplot.
import plotly.offline as py
import plotly.graph_objs as go
import plotly.figure_factory as ff
py.init_notebook_mode(connected=True)

# matplotlib appears exactly once, at the very end, to write the summary PNG.
# 'Agg' keeps it off the screen: nothing in this notebook renders through it.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy.stats import norm
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score

SEED = 1234

# Run from code/ (nbconvert's default) or from the repository root: both work.
_here = Path.cwd()
REPO = _here if (_here / 'data').is_dir() else _here.parent
DATA = REPO / 'data' / 'WA_Fn-UseC_-HR-Employee-Attrition.csv'
IMG = REPO / 'img'

BLUE, ORANGE, GREEN = '#2a78d6', '#eb6834', '#1baf7a'
AMBER, RED, GRAY, INK = '#fab219', '#d03b3b', '#8a8f98', '#333333'

print('dataset  :', DATA.name, '| found:', DATA.is_file())
print('image dir:', IMG.name, '| found:', IMG.is_dir())

dataset  : WA_Fn-UseC_-HR-Employee-Attrition.csv | found: True
image dir: img | found: True


## Rebuilding the model under audit

Auditing a model you did not fit is auditing a rumour. So the first job is to reproduce the
main notebook's winning configuration here: the same cleaning, the same 80/20 split at
`random_state=1234`, the same post-split feature engineering, the same one-hot encoding,
and the same 5-fold grid search over `C`, `l1_ratio` and class weight.

Two things to watch while it runs.

**The split happens before the engineered columns exist.** That is the main notebook's
choice and it is the right one, because in production those columns do not exist until the
row arrives.

**I take a copy of `Gender`, `MaritalStatus`, `Department` and `Age` off the test rows
before any encoding, and the model never sees that copy.** The audit needs raw labels; the
model needs encoded ones. Keeping them in two separate objects is the cheapest way to be
certain the audit is reading the data rather than reading the model's own output back to
itself.

In [2]:
data_df = pd.read_csv(DATA)

# The main notebook's cleaning, in its order: drop the constant columns, drop the
# identifier, then map the ordinal codes back to their labels.
constant_cols = [c for c in data_df.columns if data_df[c].nunique() == 1]
data_df.drop(constant_cols, axis=1, inplace=True)
data_df.drop('EmployeeNumber', axis=1, inplace=True)

four_point = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}
ordinal_maps = {
    'Education': {
        1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Phd'
    },
    'EnvironmentSatisfaction': four_point,
    'JobInvolvement': four_point,
    'JobSatisfaction': four_point,
    'PerformanceRating': {
        1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'
    },
    'RelationshipSatisfaction': four_point,
    'WorkLifeBalance': {
        1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'
    },
}
for column, mapping in ordinal_maps.items():
    data_df[column] = data_df[column].apply(lambda v, m=mapping: m[v])

X = data_df.loc[:, data_df.columns != 'Attrition']
y = data_df.loc[:, 'Attrition']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.80, random_state=SEED
)
target_map = {'Yes': 1, 'No': 0}
y_train = y_train.apply(lambda v: target_map[v])
y_test = y_test.apply(lambda v: target_map[v])

# Held aside before the pipeline touches these columns. The model never reads it.
audit_df = X_test.loc[:, ['Gender', 'MaritalStatus', 'Department', 'Age']].copy()

print('dropped as constant : %s' % constant_cols)
print('train %s, test %s' % (X_train.shape, X_test.shape))
print(
    'leavers: %d of %d in train, %d of %d in test' % (
        y_train.sum(), len(y_train), y_test.sum(), len(y_test)
    )
)

dropped as constant : ['EmployeeCount', 'Over18', 'StandardHours']
train (1176, 30), test (294, 30)
leavers: 190 of 1176 in train, 47 of 294 in test


In [3]:
def create_generation_feature(age_val: int) -> str:
    """Age to generation label, cut exactly where the main notebook cuts it."""
    if age_val < 37:
        return 'Millenials'
    if age_val < 54:
        return 'Generation X'
    if age_val < 73:
        return 'Boomers'
    return 'Silent'


def create_job_hop_index(df, total_exp_col, num_prev_com_col):
    """First-job flag, and years of experience per previous employer."""
    first_job_ind = np.where(df[num_prev_com_col] == 0, 1, 0)
    job_hop_idx = np.where(
        df[num_prev_com_col] == 0, 0.,
        df[total_exp_col] / df[num_prev_com_col]
    )
    return first_job_ind, job_hop_idx


_LOOKUP_COL = ['Department', 'JobRole', 'JobLevel']
_MEDIAN_INCOME_LOOKUP = data_df.groupby(_LOOKUP_COL). \
    agg({'MonthlyIncome': np.median, 'Age': 'count'}).reset_index(drop=False). \
    rename(columns={'MonthlyIncome': 'MedianIncome', 'Age': 'Count'})


def compute_compa_ratio_feature(df, salary_col):
    """Each employee's income over the median for their department, role and level."""
    merged = df.reset_index().merge(_MEDIAN_INCOME_LOOKUP, on=_LOOKUP_COL,
                                    how='left').set_index('index')
    merged['compa_ratio'] = merged[salary_col] / merged['MedianIncome']
    merged.drop(['MedianIncome', 'Count'], axis=1, inplace=True)
    return merged


def engineer(df):
    df = df.copy()
    df['Generation'] = df.Age.apply(create_generation_feature)
    df['First_job_ind'], df['Job_hop_idx'] = create_job_hop_index(
        df, 'TotalWorkingYears', 'NumCompaniesWorked')
    df = compute_compa_ratio_feature(df, 'MonthlyIncome')
    df.drop(
        ['Age', 'TotalWorkingYears', 'NumCompaniesWorked'], 
        axis=1, inplace=True
    )
    return df


X_train, X_test = engineer(X_train), engineer(X_test)

object_cols = [
    c for c in X_train.columns if pd.api.types.is_string_dtype(X_train[c])
]
numeric_cols = X_train.columns.difference(object_cols)


def encoding_feature(df):
    categorical = pd.DataFrame()
    for column in object_cols:
        dummies = pd.get_dummies(
            data = df.loc[:, column], 
            columns=column,
            prefix=column,
            drop_first=True,
            dtype='uint8'
        )
        categorical = pd.concat([categorical, dummies], axis=1)
    return pd.concat([df.loc[:, numeric_cols], categorical], axis=1)


X_train, X_test = encoding_feature(X_train), encoding_feature(X_test)
print('encoded: train %s, test %s' % (X_train.shape, X_test.shape))

encoded: train (1176, 59), test (294, 59)


In [4]:
cv_params = {
    'C': [0.001, 0.01, 0.1, 1., 10., 100.],
    'l1_ratio': [1.0, 0.0],   # 1.0 is l1, 0.0 is l2
    'class_weight': [None, 'balanced']
}
fix_params = {'random_state': SEED, 'solver': 'liblinear'}

grid = GridSearchCV(
    LogisticRegression(**fix_params), cv_params,
    scoring='f1', cv=5
)
grid.fit(X_train, y_train)

model = LogisticRegression(**{**fix_params, **grid.best_params_})
model.fit(X_train, y_train)

y_true = y_test.to_numpy()
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]


def confusion_cells(actual, predicted):
    """(tn, fp, fn, tp) as plain ints. Every metric below is built from these."""
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    return (int(((actual == 0) & (predicted == 0)).sum()),
            int(((actual == 0) & (predicted == 1)).sum()),
            int(((actual == 1) & (predicted == 0)).sum()),
            int(((actual == 1) & (predicted == 1)).sum()))


TN, FP, FN, TP = confusion_cells(y_true, y_pred)
print('grid winner        : %s' % grid.best_params_)
print('confusion tn fp fn tp: %d, %d, %d, %d' % (TN, FP, FN, TP))
print('accuracy %.4f, precision %.4f, recall %.4f, F1 %.4f'
      % ((TP + TN) / len(y_true), TP / (TP + FP), TP / (TP + FN),
         2 * TP / (2 * TP + FP + FN)))

# The audit below is only about this model if this model is the one the README
# describes. If a library upgrade moves it, stop here rather than mislead.
assert (TN, FP, FN, TP) == (236, 11, 20, 27), 'the model under audit moved'
assert grid.best_params_ == {'C': 100.0, 'class_weight': None,
                             'l1_ratio': 1.0}, 'the grid picked something else'
print('\nmatches the README headline model')

grid winner        : {'C': 100.0, 'class_weight': None, 'l1_ratio': 1.0}
confusion tn fp fn tp: 236, 11, 20, 27
accuracy 0.8946, precision 0.7105, recall 0.5745, F1 0.6353

matches the README headline model


The confusion matrix comes back 236, 11, 20, 27, which is the matrix the README quotes for
the headline model, and the grid lands on `C=100`, `l1_ratio=1.0`, no class weighting, which
is the configuration the README names. The two `assert` lines are not decoration. If a
library upgrade nudges this model, everything below would quietly become an audit of a
different classifier, and a notebook that stops is better than one that misleads.

Now look at what the model is actually reading.

In [5]:
sensitive_like = [
    c for c in X_train.columns
        if c.startswith(('Gender', 'MaritalStatus', 'Department', 'Generation'))
]
print('columns in the model that encode who someone is, rather than what they do:')
for column in sensitive_like:
    print(
        '  %-34s coefficient % .4f' % (
            column, model.coef_[0][list(X_train.columns).index(column)]
        )
    )
print('\n%d of the %d features. Age itself was dropped, but only after being '
      'recoded\nas Generation, so age is in the model at three-bucket '
      'resolution.' % (len(sensitive_like), X_train.shape[1]))

columns in the model that encode who someone is, rather than what they do:
  Department_Research & Development  coefficient  1.1606
  Department_Sales                   coefficient  1.1252
  Gender_Male                        coefficient  0.3814
  MaritalStatus_Married              coefficient  0.5679
  MaritalStatus_Single               coefficient  1.5596
  Generation_Generation X            coefficient -0.8392
  Generation_Millenials              coefficient -0.0131

7 of the 59 features. Age itself was dropped, but only after being recoded
as Generation, so age is in the model at three-bucket resolution.


Seven of the 59 features describe who someone is rather than what they do or what the job
pays them. Nothing in the main notebook flags that, and nothing in scikit-learn will. The
model is not doing anything wrong by fitting them: `MaritalStatus_Single` carries real
signal in this dataset. The question a fairness audit asks is not "did the model use these"
but "what does using them do to people, group by group".

That question needs groups. On to Part 1.

---

# Part 1: the audit

## 1.1 Who is in the room

Every fairness metric is a comparison between groups, so the first number in any audit is
the size of each group. Not because size is interesting, but because it is the ceiling on
how much any later number can mean. A rate computed on eleven people is a rate computed on
eleven people no matter how many decimal places pandas prints.

### Defining the age band, and saying why

Gender and marital status arrive as categories. Age arrives as an integer from 18 to 60, so
I have to draw the line myself, and where I draw it changes the answer. That makes it a
decision to defend rather than a default to accept.

I use one cut, at 40, for three reasons:

1. **It is the boundary that carries legal weight in the jurisdiction this dataset
   imitates.** The US Age Discrimination in Employment Act protects employees aged 40 and
   over. A cut somewhere else measures something real but answers a question nobody is
   going to ask you.
2. **It leaves both sides large enough to say anything at all.** The cell below prints the
   counts so you can check that rather than take it from me.
3. **Finer bands collapse.** The same cell prints a four-band version, and you can watch
   the leaver counts drop into single digits, at which point the group's true positive rate
   is arithmetic over a handful of people.

The main notebook already made this decision once without announcing it: it converts `Age`
into `Generation` at 37 and 54. Any age band is a modelling choice with consequences. The
honest move is to print the counts beside it.

In [6]:
AGE_CUT = 40
audit_df['AgeBand'] = np.where(audit_df.Age >= AGE_CUT, '40 and over', 'Under 40')
audit_df['actual'] = y_true
audit_df['predicted'] = y_pred
audit_df['score'] = y_score

print('the cut I use, at %d:' % AGE_CUT)
for label in ['Under 40', '40 and over']:
    sub = audit_df[audit_df.AgeBand == label]
    print('  %-12s n=%3d  leavers=%2d' % (label, len(sub), sub.actual.sum()))

print('\nthe same test set in four bands, for comparison:')
finer = pd.cut(audit_df.Age, [17, 29, 39, 49, 60],
               labels=['18 to 29', '30 to 39', '40 to 49', '50 to 60'])
for label, sub in audit_df.groupby(finer, observed=True):
    print('  %-12s n=%3d  leavers=%2d' % (label, len(sub), sub.actual.sum()))
print('\nThe oldest band holds 2 leavers. A true positive rate over 2 people can '
      'only\ntake three values, and none of them is evidence of anything.')

the cut I use, at 40:
  Under 40     n=174  leavers=35
  40 and over  n=120  leavers=12

the same test set in four bands, for comparison:
  18 to 29     n= 59  leavers=17
  30 to 39     n=115  leavers=18
  40 to 49     n= 91  leavers=10
  50 to 60     n= 29  leavers= 2

The oldest band holds 2 leavers. A true positive rate over 2 people can only
take three values, and none of them is evidence of anything.


Two bands it is. Now the representation table: how many employees sit in each group, and
what share of them actually left. That second number is the **base rate**, and it is the
single most important column in this notebook, because most of what looks like model bias
in a first audit is the model faithfully reproducing a base rate difference that was
already in the data.

In [7]:
DIMENSIONS = ['Gender', 'MaritalStatus', 'AgeBand', 'Department']

representation = []
for dimension in DIMENSIONS:
    for level in sorted(audit_df[dimension].unique()):
        sub = audit_df[audit_df[dimension] == level]
        representation.append({'dimension': dimension, 'group': level,
                               'n': len(sub), 'leavers': int(sub.actual.sum()),
                               'share_of_test': len(sub) / len(audit_df),
                               'base_rate': sub.actual.mean()})
representation = pd.DataFrame(representation)
print(representation.to_string(index=False,
                               float_format=lambda v: format(v, '.4f')))

    dimension                  group   n  leavers  share_of_test  base_rate
       Gender                 Female 109       14         0.3707     0.1284
       Gender                   Male 185       33         0.6293     0.1784
MaritalStatus               Divorced  60        8         0.2041     0.1333
MaritalStatus                Married 144       18         0.4898     0.1250
MaritalStatus                 Single  90       21         0.3061     0.2333
      AgeBand            40 and over 120       12         0.4082     0.1000
      AgeBand               Under 40 174       35         0.5918     0.2011
   Department        Human Resources  11        1         0.0374     0.0909
   Department Research & Development 185       24         0.6293     0.1297
   Department                  Sales  98       22         0.3333     0.2245


Read the `base_rate` column before anything else.

Single employees leave at roughly twice the rate of married ones in this file. Employees
under 40 leave at twice the rate of those over. Human Resources is eleven people. None of
that is the model's doing: it is what IBM's generator produced, and any model fitted on it
will reproduce those differences because reproducing them is what being accurate means
here.

That is the trap in the first metric we are about to compute, and it is worth naming before
rather than after.

In [8]:
def hover_bars(frame, value_col, title, y_title, colour):
    """One bar per group, with n and leavers carried in the hover text."""
    detail = np.stack([frame.n.to_numpy(), frame.leavers.to_numpy()], axis=-1)
    labels = [f'{d}<br>{g}' for d, g in zip(frame.dimension, frame.group)]
    bar = go.Bar(x=labels, y=frame[value_col], customdata=detail,
                 name=value_col,
                 marker=dict(color=colour, line=dict(color=INK, width=0.6)),
                 hovertemplate=('<b>%{x}</b><br>' + y_title + ': %{y:.4f}'
                                '<br>group size: %{customdata[0]}'
                                '<br>leavers in group: %{customdata[1]}'
                                '<extra></extra>'))
    layout = go.Layout(dict(title=dict(text=title, font=dict(size=15)),
                            height=430, width=880,
                            yaxis=dict(title=y_title, tickformat='.2f'),
                            xaxis=dict(tickfont=dict(size=10)),
                            plot_bgcolor='rgba(240, 240, 240, 0.95)',
                            paper_bgcolor='rgba(240, 240, 240, 0.95)',
                            showlegend=False))
    return go.Figure(data=[bar], layout=layout)


py.iplot(hover_bars(representation, 'base_rate',
                    'Base attrition rate by group, 294 held-out employees',
                    'base rate', BLUE))

Hover over any bar and it tells you the group size and the leaver count behind the rate,
which is the thing a static bar chart cannot do and the reason these charts are plotly. The Human
Resources bar looks like a finding until the hover says `group size: 11`.

## 1.2 Selection rate, and demographic parity

**Selection rate** is the share of a group the model flags. Here, being flagged means being
put on a retention list, which is a benefit: attention from a manager, possibly a
counter-offer, possibly a transfer. A group flagged less often receives less of that
benefit.

**Demographic parity** asks for those selection rates to be equal. The usual summary is the
**demographic parity ratio**: the worst group's selection rate divided by the best group's.
1.0 means identical treatment, and lower is worse.

Everything from here is built from four numbers per group, so compute them once and reuse
them. The full audit table is the confusion matrix of the same model, sliced.

In [9]:
audit_rows = []
for dimension in DIMENSIONS:
    for level in sorted(audit_df[dimension].unique()):
        sub = audit_df[audit_df[dimension] == level]
        tn, fp, fn, tp = confusion_cells(sub.actual, sub.predicted)
        audit_rows.append({'dimension': dimension, 'group': level,
                           'n': tn + fp + fn + tp, 'leavers': fn + tp,
                           'flagged': fp + tp,
                           'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp})
audit = pd.DataFrame(audit_rows)

print('AUDIT TABLE: the model\'s confusion matrix, sliced by group')
print(audit.to_string(index=False))
print('\nEvery rate in this notebook is derived from these seven integer columns.')

AUDIT TABLE: the model's confusion matrix, sliced by group
    dimension                  group   n  leavers  flagged  tn  fp  fn  tp
       Gender                 Female 109       14       10  92   3   7   7
       Gender                   Male 185       33       28 144   8  13  20
MaritalStatus               Divorced  60        8        4  51   1   5   3
MaritalStatus                Married 144       18       16 121   5   7  11
MaritalStatus                 Single  90       21       18  64   5   8  13
      AgeBand            40 and over 120       12        8 103   5   9   3
      AgeBand               Under 40 174       35       30 133   6  11  24
   Department        Human Resources  11        1        1  10   0   0   1
   Department Research & Development 185       24       24 153   8   8  16
   Department                  Sales  98       22       13  73   3  12  10

Every rate in this notebook is derived from these seven integer columns.


In [10]:
# Derived rates. Nothing here reads the data again: it is all the table above.
audit['base_rate'] = audit.leavers / audit.n
audit['selection_rate'] = audit.flagged / audit.n
audit['tpr'] = audit.tp / audit.leavers                  # equal opportunity
audit['fpr'] = audit.fp / (audit.fp + audit.tn)          # the other half of odds
audit['ppv'] = np.where(audit.flagged > 0, audit.tp / audit.flagged, np.nan)

shown = ['dimension', 'group', 'n', 'base_rate', 'selection_rate', 'tpr',
         'fpr', 'ppv']
print(audit[shown].to_string(index=False,
                             float_format=lambda v: format(v, '.4f')))

    dimension                  group   n  base_rate  selection_rate    tpr    fpr    ppv
       Gender                 Female 109     0.1284          0.0917 0.5000 0.0316 0.7000
       Gender                   Male 185     0.1784          0.1514 0.6061 0.0526 0.7143
MaritalStatus               Divorced  60     0.1333          0.0667 0.3750 0.0192 0.7500
MaritalStatus                Married 144     0.1250          0.1111 0.6111 0.0397 0.6875
MaritalStatus                 Single  90     0.2333          0.2000 0.6190 0.0725 0.7222
      AgeBand            40 and over 120     0.1000          0.0667 0.2500 0.0463 0.3750
      AgeBand               Under 40 174     0.2011          0.1724 0.6857 0.0432 0.8000
   Department        Human Resources  11     0.0909          0.0909 1.0000 0.0000 1.0000
   Department Research & Development 185     0.1297          0.1297 0.6667 0.0497 0.6667
   Department                  Sales  98     0.2245          0.1327 0.4545 0.0395 0.7692


In [11]:
def parity_ratio(table, dimension, column):
    """Worst group over best group, on one dimension, plus who they are."""
    sub = table[table.dimension == dimension]
    best = sub.loc[sub[column].idxmax()]
    worst = sub.loc[sub[column].idxmin()]
    return worst[column] / best[column], worst.group, best.group


print('demographic parity ratio, worst group over best:')
for dimension in DIMENSIONS:
    ratio, low, high = parity_ratio(audit, dimension, 'selection_rate')
    sub = audit[audit.dimension == dimension]
    print('  %-14s %.4f   %s (%.4f) against %s (%.4f)'
          % (dimension, ratio, low,
             sub.loc[sub.group == low, 'selection_rate'].iloc[0], high,
             sub.loc[sub.group == high, 'selection_rate'].iloc[0]))

demographic parity ratio, worst group over best:
  Gender         0.6062   Female (0.0917) against Male (0.1514)
  MaritalStatus  0.3333   Divorced (0.0667) against Single (0.2000)
  AgeBand        0.3867   40 and over (0.0667) against Under 40 (0.1724)
  Department     0.6853   Human Resources (0.0909) against Sales (0.1327)


## 1.3 The four-fifths rule

The oldest rule of thumb in this field, and the only one in this notebook with a citation
behind it. The US Uniform Guidelines on Employee Selection Procedures, 29 CFR 1607.4(D),
treat a selection rate for any group that is less than four-fifths (80%) of the rate for the
highest group as evidence of adverse impact worth investigating.

Three things about it that people skip:

- **It is a screening heuristic, not a verdict.** The guidelines say "generally regarded as
  evidence", and the same text notes that greater differences may not matter when the
  numbers are small and smaller differences may matter when they are large. It starts an
  investigation. It does not end one.
- **It was written about hiring and promotion**, not about an internal model that picks who
  gets a retention conversation. I am applying it by analogy because the shape is the same:
  a scarce benefit, allocated by a rule, at different rates to different groups.
- **It ignores base rates entirely**, which is the objection you will meet within thirty
  seconds of showing it to anyone. Hold that thought until the cell after the verdict.

In [12]:
FOUR_FIFTHS = 0.80

print('four-fifths rule, threshold %.2f' % FOUR_FIFTHS)
verdicts = {}
for dimension in DIMENSIONS:
    ratio, low, high = parity_ratio(audit, dimension, 'selection_rate')
    verdicts[dimension] = ratio >= FOUR_FIFTHS
    print('  %-14s ratio %.4f  ->  %s   (%s vs %s)'
          % (dimension, ratio, 'PASSES' if verdicts[dimension] else 'FAILS',
             low, high))

failed = [d for d, ok in verdicts.items() if not ok]
print('\nFails on %d of the %d dimensions audited: %s.'
      % (len(failed), len(DIMENSIONS), ', '.join(failed)))

four-fifths rule, threshold 0.80
  Gender         ratio 0.6062  ->  FAILS   (Female vs Male)
  MaritalStatus  ratio 0.3333  ->  FAILS   (Divorced vs Single)
  AgeBand        ratio 0.3867  ->  FAILS   (40 and over vs Under 40)
  Department     ratio 0.6853  ->  FAILS   (Human Resources vs Sales)

Fails on 4 of the 4 dimensions audited: Gender, MaritalStatus, AgeBand, Department.


In [13]:
best_by_dim = audit.groupby('dimension').selection_rate.transform('max')
plot = audit.assign(four_fifths_line=best_by_dim * FOUR_FIFTHS)
detail = np.stack([plot.n, plot.flagged, plot.base_rate], axis=-1)
labels = [f'{d}<br>{g}' for d, g in zip(plot.dimension, plot.group)]

bars = go.Bar(x=labels, y=plot.selection_rate, customdata=detail,
              name='selection rate',
              marker=dict(color=ORANGE, line=dict(color=INK, width=0.6)),
              hovertemplate=('<b>%{x}</b><br>selection rate: %{y:.4f}'
                             '<br>group size: %{customdata[0]}'
                             '<br>flagged: %{customdata[1]}'
                             '<br>base attrition rate: %{customdata[2]:.4f}'
                             '<extra></extra>'))
line = go.Scatter(x=labels, y=plot.four_fifths_line, mode='markers',
                  name='four-fifths of the best group in that dimension',
                  marker=dict(color=RED, symbol='line-ew-open', size=26,
                              line=dict(width=3, color=RED)),
                  hovertemplate=('four-fifths line: %{y:.4f}<extra></extra>'))
layout = go.Layout(dict(
    title=dict(text='Selection rate by group, against the four-fifths line',
               font=dict(size=15)),
    height=470, width=880, yaxis=dict(title='share of the group flagged'),
    xaxis=dict(tickfont=dict(size=10)),
    plot_bgcolor='rgba(240, 240, 240, 0.95)',
    paper_bgcolor='rgba(240, 240, 240, 0.95)',
    legend=dict(orientation='h', y=-0.28)))
py.iplot(go.Figure(data=[bars, line], layout=layout))

Every bar that falls below its red marker fails the rule on that dimension, and on this
model every one of the four does.

**Department is in the table for contrast, not as a protected class.** Nobody argues that a
model should flag Sales and Research at equal rates: those departments do different work and
lose people at different rates for reasons an employer is entitled to act on. It fails the
same screen anyway, at 0.6853, and the group dragging it down is Human Resources with eleven
people in it. Both facts are worth having. The first shows how readily this rule fires on
something nobody is worried about, and the second is the thing to check before quoting any
ratio at all.

And now the objection. Under 40, the base attrition rate is roughly twice what it is at 40
and over. A model that flagged both bands at the same rate would be *wrong* about one of
them. Demographic parity, taken literally, asks the model to be inaccurate. That is why
nobody who does this for a living stops at demographic parity: it is a screening question
that tells you where to look, and the metric that decides whether something is broken is
the next one.

Keep the failure in mind, though. It is the number a regulator or a works council will ask
about first, and "our base rates differ" is the beginning of an answer, not the whole of
one, because base rates in an HR dataset are themselves the output of earlier decisions.

## 1.4 Equal opportunity: the metric that matters here

**Equal opportunity** compares the **true positive rate** across groups: of the people in
this group who really did leave, what share did the model catch?

This is the one that matters in a retention setting, and the reason is worth stating in
plain words. A missed leaver is a retention conversation that never happened. If the model
catches 70% of one group's leavers and 25% of another's, then the second group is being
quietly excluded from the intervention the model exists to trigger. They resign, and nobody
ever asked them to stay.

Note what this metric is conditioned on: *among people who actually left*. Base rate
differences drop out. That is exactly the objection demographic parity could not answer.

In [14]:
print('true positive rate: leavers caught, out of leavers in the group')
for dimension in DIMENSIONS:
    sub = audit[audit.dimension == dimension]
    print('  %s' % dimension)
    for _, r in sub.iterrows():
        print('    %-24s %2d of %2d = %.4f'
              % (r.group, r.tp, r.leavers, r.tpr))
    ratio, low, high = parity_ratio(audit, dimension, 'tpr')
    print('    ratio %.4f (%s over %s)' % (ratio, low, high))

true positive rate: leavers caught, out of leavers in the group
  Gender
    Female                    7 of 14 = 0.5000
    Male                     20 of 33 = 0.6061
    ratio 0.8250 (Female over Male)
  MaritalStatus
    Divorced                  3 of  8 = 0.3750
    Married                  11 of 18 = 0.6111
    Single                   13 of 21 = 0.6190
    ratio 0.6058 (Divorced over Single)
  AgeBand
    40 and over               3 of 12 = 0.2500
    Under 40                 24 of 35 = 0.6857
    ratio 0.3646 (40 and over over Under 40)
  Department
    Human Resources           1 of  1 = 1.0000
    Research & Development   16 of 24 = 0.6667
    Sales                    10 of 22 = 0.4545
    ratio 0.4545 (Sales over Human Resources)


There it is. The model reaches 24 of the 35 leavers under 40 and 3 of the 12 leavers aged
40 and over. On gender the gap is much smaller. On marital status it sits in between, and on
department it is an artefact of a group with one leaver in it.

Before writing any of that down as a finding, we have to answer the question that makes or
breaks a small-sample audit: how much of this could be luck?

## 1.5 The intervals, computed rather than caveated

Twelve leavers. A true positive rate of 3/12 moves by 0.083 every time one person changes
category. Quoting it to four decimal places, as I just did, is a way of hiding that.

So compute the interval. I use the **Wilson score interval**, which is a few lines of
arithmetic and behaves properly at small counts and at rates near 0 or 1, where the
textbook normal approximation produces intervals running past 100%.

The `z` below is the normal quantile for a 95% interval. I take it from scipy rather than
typing 1.96, so the confidence level is a parameter of this notebook and not a magic number
inside it.

In [15]:
CONFIDENCE = 0.95
Z = norm.ppf(1 - (1 - CONFIDENCE) / 2)
print('z for a %.0f%% interval: %.6f' % (CONFIDENCE * 100, Z))


def wilson(successes, trials, z=Z):
    """Wilson score interval for a proportion. Two lines of algebra, no library."""
    if trials == 0:
        return (np.nan, np.nan)
    p = successes / trials
    denominator = 1 + z ** 2 / trials
    centre = (p + z ** 2 / (2 * trials)) / denominator
    half = z * np.sqrt(p * (1 - p) / trials
                       + z ** 2 / (4 * trials ** 2)) / denominator
    return centre - half, centre + half


audit['tpr_lo'], audit['tpr_hi'] = zip(*[wilson(r.tp, r.leavers)
                                         for _, r in audit.iterrows()])
audit['sel_lo'], audit['sel_hi'] = zip(*[wilson(r.flagged, r.n)
                                         for _, r in audit.iterrows()])

print('\ntrue positive rate with a %.0f%% Wilson interval:' % (CONFIDENCE * 100))
for _, r in audit.iterrows():
    print('  %-14s %-24s %2d/%-2d = %.4f  [%.4f, %.4f]  width %.4f'
          % (r.dimension, r.group, r.tp, r.leavers, r.tpr, r.tpr_lo, r.tpr_hi,
             r.tpr_hi - r.tpr_lo))

z for a 95% interval: 1.959964

true positive rate with a 95% Wilson interval:
  Gender         Female                    7/14 = 0.5000  [0.2680, 0.7320]  width 0.4640
  Gender         Male                     20/33 = 0.6061  [0.4368, 0.7532]  width 0.3163
  MaritalStatus  Divorced                  3/8  = 0.3750  [0.1368, 0.6943]  width 0.5574
  MaritalStatus  Married                  11/18 = 0.6111  [0.3862, 0.7969]  width 0.4108
  MaritalStatus  Single                   13/21 = 0.6190  [0.4088, 0.7925]  width 0.3837
  AgeBand        40 and over               3/12 = 0.2500  [0.0889, 0.5323]  width 0.4434
  AgeBand        Under 40                 24/35 = 0.6857  [0.5202, 0.8145]  width 0.2943
  Department     Human Resources           1/1  = 1.0000  [0.2065, 1.0000]  width 0.7935
  Department     Research & Development   16/24 = 0.6667  [0.4671, 0.8203]  width 0.3532
  Department     Sales                    10/22 = 0.4545  [0.2692, 0.6534]  width 0.3842


In [16]:
tpr_plot = audit[audit.dimension.isin(['Gender', 'MaritalStatus',
                                       'AgeBand'])].copy()
labels = [f'{d}: {g}' for d, g in zip(tpr_plot.dimension, tpr_plot.group)]
detail = np.stack([tpr_plot.tp, tpr_plot.leavers, tpr_plot.tpr_lo,
                   tpr_plot.tpr_hi], axis=-1)

points = go.Scatter(
    x=tpr_plot.tpr, y=labels, mode='markers', name='true positive rate',
    marker=dict(color=BLUE, size=12, line=dict(color=INK, width=1)),
    error_x=dict(type='data', symmetric=False,
                 array=tpr_plot.tpr_hi - tpr_plot.tpr,
                 arrayminus=tpr_plot.tpr - tpr_plot.tpr_lo,
                 color=GRAY, thickness=1.6, width=7),
    hovertemplate=('<b>%{y}</b><br>caught %{customdata[0]} of '
                   '%{customdata[1]} leavers<br>TPR: %{x:.4f}'
                   '<br>95% interval: [%{customdata[2]:.4f}, '
                   '%{customdata[3]:.4f}]<extra></extra>'),
    customdata=detail)
layout = go.Layout(dict(
    title=dict(text='Equal opportunity, with the uncertainty drawn in',
               font=dict(size=15)),
    height=440, width=880,
    xaxis=dict(title='true positive rate, share of a group\'s leavers caught',
               range=[0, 1.02]),
    yaxis=dict(automargin=True),
    plot_bgcolor='rgba(240, 240, 240, 0.95)',
    paper_bgcolor='rgba(240, 240, 240, 0.95)', showlegend=False))
py.iplot(go.Figure(data=[points], layout=layout))

This chart is the honest version of the table above it. The dots are the same rates; the
whiskers are what 294 people can support. On this dataset the intervals are wider than most
of the gaps they are being used to argue about.

Which raises the question people usually get wrong: **do overlapping intervals mean there
is no difference?** No. Comparing two intervals is a conservative test, and it will tell you
"no difference" in cases where a proper test of the difference says otherwise. The thing to
put an interval on is the gap itself.

For a difference of two proportions, the method that agrees with Wilson is Newcombe's, and
it is three lines: take each group's Wilson interval and combine the distances from each
point estimate to the relevant end.

In [17]:
def newcombe(k1, n1, k2, n2):
    """Interval for p1 - p2, built from the two Wilson intervals."""
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = wilson(k1, n1)
    l2, u2 = wilson(k2, n2)
    lower = (p1 - p2) - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2)
    upper = (p1 - p2) + np.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)
    return lower, upper


def tpr_gap(dimension):
    sub = audit[audit.dimension == dimension]
    high = sub.loc[sub.tpr.idxmax()]
    low = sub.loc[sub.tpr.idxmin()]
    lo, hi = newcombe(high.tp, high.leavers, low.tp, low.leavers)
    return high, low, high.tpr - low.tpr, lo, hi


print(
    'gap in true positive rate, with a %.0f%% interval on the gap itself:' % (
        CONFIDENCE * 100
    )
)
gaps = {}
for dimension in ['Gender', 'MaritalStatus', 'AgeBand']:
    high, low, gap, lo, hi = tpr_gap(dimension)
    real = lo > 0 or hi < 0
    gaps[dimension] = dict(
        high=high.group, low=low.group, gap=gap, lo=lo,
        hi=hi, excludes_zero=bool(real)
    )
    print(
        '  %-14s %s (%.4f) minus %s (%.4f) = %.4f  [%.4f, %.4f]  %s' % (
            dimension, high.group, high.tpr, low.group, low.tpr, gap, lo, hi,
            'excludes zero' if real else 'contains zero'
        )
    )

gap in true positive rate, with a 95% interval on the gap itself:
  Gender         Male (0.6061) minus Female (0.5000) = 0.1061  [-0.1811, 0.3808]  contains zero
  MaritalStatus  Single (0.6190) minus Divorced (0.3750) = 0.2440  [-0.1382, 0.5387]  contains zero
  AgeBand        Under 40 (0.6857) minus 40 and over (0.2500) = 0.4357  [0.1085, 0.6419]  excludes zero


This is the result of the audit, and it is narrower than the table of rates suggested.

- **Gender.** The gap is 0.1061 and its interval runs from -0.1811 to 0.3808. It contains
  zero. On 14 female leavers and 33 male ones, this dataset cannot tell you whether the
  model treats them differently. Reporting "the model catches 61% of male leavers and 50% of
  female ones" without that interval would be reporting noise as a finding.
- **Marital status.** Same verdict, wider: 0.2440, interval -0.1382 to 0.5387.
- **Age.** 0.4357, interval 0.1085 to 0.6419. It excludes zero. This is the one gap in the
  audit that survives its own uncertainty.

Note also that the age intervals *overlap* in the previous chart ([0.089, 0.532] against
[0.520, 0.815], touching at the ends) while the interval on the difference does not contain
zero. That is the conservative-test problem, live, on this data. If you compare error bars
by eye, this is the finding you would have thrown away.

**And the reminder, because this is the point where it is easiest to forget.** These are
invented employees. What has been demonstrated is that this procedure can find and size a disparate true positive rate, and that it correctly declines to call the other two. Nothing has been demonstrated about anybody's real workforce.

## 1.6 Equalised odds: both error rates at once

Equal opportunity only looks at the people who left. **Equalised odds** adds the other row
of the confusion matrix and asks for the true positive rate *and* the false positive rate to
match across groups.

The false positive rate matters because a retention conversation is not free and not always
welcome. A group with a high false positive rate is being pulled into conversations about
leaving that it was not having. In some settings that is a nuisance; in others, being
labelled a flight risk follows you into the next promotion round.

Plotted as a point per group, equalised odds is the statement that all the points sit on top
of each other.

In [18]:
odds = audit[audit.dimension.isin(['Gender', 'MaritalStatus', 'AgeBand'])]
traces = []
for colour, dimension in zip(
    [BLUE, ORANGE, GREEN],
    ['Gender', 'MaritalStatus', 'AgeBand']
):
    sub = odds[odds.dimension == dimension]
    detail = np.stack([sub.n, sub.leavers, sub.tp, sub.fp], axis=-1)
    traces.append(go.Scatter(
        x=sub.fpr, y=sub.tpr, mode='markers+text', name=dimension,
        text=sub.group, textposition='top center',
        textfont=dict(size=10, color=INK),
        marker=dict(color=colour, size=15, line=dict(color=INK, width=1)),
        customdata=detail,
        hovertemplate=(
            '<b>%{text}</b><br>TPR: %{y:.4f}<br>FPR: %{x:.4f}'
            '<br>group size: %{customdata[0]}'
            '<br>leavers: %{customdata[1]}'
            '<br>caught: %{customdata[2]}, false alarms: '
            '%{customdata[3]}<extra></extra>')
    ))

layout = go.Layout(dict(
    title=dict(text='Equalised odds: every group should be one point',
               font=dict(size=15)),
    height=520, width=880,
    xaxis=dict(title='false positive rate, stayers wrongly flagged',
               range=[0, 0.11]),
    yaxis=dict(title='true positive rate, leavers caught', range=[0, 0.85]),
    plot_bgcolor='rgba(240, 240, 240, 0.95)',
    paper_bgcolor='rgba(240, 240, 240, 0.95)',
    legend=dict(orientation='h', y=-0.2)))
py.iplot(go.Figure(data=traces, layout=layout))

The spread is almost entirely vertical. False positive rates sit in a narrow band from 0.019
to 0.073, while true positive rates run from 0.25 to 0.69. In other words this model's
groups differ in who it *finds*, not in who it *bothers*. That is a useful thing to be able to say in one sentence to a manager, and this chart is where it comes from.

It also explains why equalised odds and equal opportunity agree here. They do not always. A
model can equalise true positive rates by flagging one group far more aggressively, buying
recall with false alarms, and the vertical-only spread is what tells you that is not what is
happening.

## 1.7 Calibration: does a score of 0.7 mean the same thing to everyone?

The metrics so far all read hard predictions. But the main notebook does not stop at hard
predictions: it ranks employees by score and works down the list, and its README says
plainly that the top band holds 8 people. Once a score is being read as a probability, it
needs to mean the same thing for every group.

**Calibration** is that check. Bucket the scores, and inside each bucket compare the average
predicted score against the share who actually left. A calibrated model sits on the diagonal.
The buckets have to be coarse here, because 294 people do not support ten of them.

In [19]:
BINS = [0.0, 0.2, 0.5, 1.0]
BIN_LABELS = ['under 0.2', '0.2 to 0.5', '0.5 and above']

calibration = []
for dimension in ['Gender', 'AgeBand']:
    for level in sorted(audit_df[dimension].unique()):
        sub = audit_df[audit_df[dimension] == level]
        buckets = pd.cut(sub.score, BINS, labels=BIN_LABELS,
                         include_lowest=True)
        for label, cell in sub.groupby(buckets, observed=True):
            calibration.append({'dimension': dimension, 'group': level,
                                'bucket': label, 'n': len(cell),
                                'mean_score': cell.score.mean(),
                                'observed': cell.actual.mean(),
                                'left': int(cell.actual.sum())})
calibration = pd.DataFrame(calibration)
print(calibration.to_string(index=False,
                            float_format=lambda v: format(v, '.4f')))

dimension       group        bucket   n  mean_score  observed  left
   Gender      Female     under 0.2  83      0.0390    0.0120     1
   Gender      Female    0.2 to 0.5  16      0.3436    0.3750     6
   Gender      Female 0.5 and above  10      0.6935    0.7000     7
   Gender        Male     under 0.2 136      0.0459    0.0662     9
   Gender        Male    0.2 to 0.5  21      0.3080    0.1905     4
   Gender        Male 0.5 and above  28      0.6992    0.7143    20
  AgeBand 40 and over     under 0.2  98      0.0392    0.0408     4
  AgeBand 40 and over    0.2 to 0.5  14      0.3218    0.3571     5
  AgeBand 40 and over 0.5 and above   8      0.6796    0.3750     3
  AgeBand    Under 40     under 0.2 121      0.0466    0.0496     6
  AgeBand    Under 40    0.2 to 0.5  23      0.3244    0.2174     5
  AgeBand    Under 40 0.5 and above  30      0.7025    0.8000    24


In [20]:
traces = [go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='perfect',
                     line=dict(color=GRAY, width=1.5, dash='dot'),
                     hoverinfo='skip')]
palette = {'Female': BLUE, 'Male': ORANGE, 'Under 40': GREEN,
           '40 and over': RED}
for (dimension, group), sub in calibration.groupby(['dimension', 'group']):
    detail = np.stack([sub.n, sub.left, [str(b) for b in sub.bucket]], axis=-1)
    traces.append(go.Scatter(
        x=sub.mean_score, y=sub.observed, mode='markers+lines',
        name=f'{dimension}: {group}',
        marker=dict(color=palette[group], size=np.sqrt(sub.n) * 2.6,
                    line=dict(color=INK, width=1)),
        line=dict(color=palette[group], width=1.4), customdata=detail,
        hovertemplate=('<b>' + str(group) + '</b>, %{customdata[2]}'
                       '<br>mean predicted score: %{x:.4f}'
                       '<br>actually left: %{y:.4f}'
                       '<br>%{customdata[1]} of %{customdata[0]} people'
                       '<extra></extra>')))

layout = go.Layout(dict(
    title=dict(text='Calibration by group (marker size is bucket size)',
               font=dict(size=15)),
    height=520, width=880,
    xaxis=dict(title='mean predicted score in the bucket', range=[0, 0.85]),
    yaxis=dict(title='share who actually left', range=[0, 0.95]),
    plot_bgcolor='rgba(240, 240, 240, 0.95)',
    paper_bgcolor='rgba(240, 240, 240, 0.95)',
    legend=dict(orientation='h', y=-0.2)))
py.iplot(go.Figure(data=traces, layout=layout))

In [21]:
top = calibration[calibration.bucket == '0.5 and above']
print('the top bucket, group by group:')
for _, r in top.iterrows():
    lo, hi = wilson(r.left, r.n)
    print('  %-12s n=%3d  mean score %.4f  actually left %.4f  '
          '[%.4f, %.4f]' % (r.group, r.n, r.mean_score, r.observed, lo, hi))

the top bucket, group by group:
  Female       n= 10  mean score 0.6935  actually left 0.7000  [0.3968, 0.8922]
  Male         n= 28  mean score 0.6992  actually left 0.7143  [0.5294, 0.8475]
  40 and over  n=  8  mean score 0.6796  actually left 0.3750  [0.1368, 0.6943]
  Under 40     n= 30  mean score 0.7025  actually left 0.8000  [0.6269, 0.9049]


Three of the four groups land near the diagonal. The fourth does not: among employees aged
40 and over who scored above 0.5, the model's average score was 0.6796 and 0.3750 of them
actually left.

And then the interval on that last number is [0.1368, 0.6943], because the bucket holds
**eight people**.

I am leaving this in, with the interval printed next to it, because the temptation here is
exactly the one this notebook exists to resist. A miscalibration of that size would be a
serious finding: it would mean a high score means something different depending on the
employee's age, and every decision made by sorting the list downward would inherit that. On
eight people, it is not a finding. It is a place to look again with more data, and the only
correct thing to write in the report is both halves of that sentence.

## 1.8 Would dropping the protected column fix any of this?

The most common first suggestion in any fairness conversation: delete `Gender`, delete
`MaritalStatus`, and the problem goes away. It usually does not, because the remaining
features carry the information anyway. The fix is called **fairness through unawareness** and
it fails whenever a proxy survives.

That is a claim you can test in about ten lines, and it costs nothing to test, so there is no excuse for asserting it either way. Take the protected attribute out of the feature matrix, fit a model to predict
*it* from everything that is left, and see how well that does on the test set.

In [22]:
def proxy_recoverability(target_train, target_test, drop_columns, label):
    """Fit the remaining features to the protected attribute itself."""
    left_train = X_train.drop(columns=drop_columns)
    left_test = X_test.drop(columns=drop_columns)
    probe = LogisticRegression(random_state=SEED, solver='liblinear', C=1.0)
    probe.fit(left_train, target_train)
    scores = probe.predict_proba(left_test)[:, 1]
    fpr, tpr, _ = roc_curve(target_test, scores)
    auc = roc_auc_score(target_test, scores)
    top = pd.Series(probe.coef_[0], index=left_train.columns)
    top = top.reindex(top.abs().sort_values(ascending=False).index)[:3]
    print('%-32s AUC %.4f  from %d features' % (label, auc, left_train.shape[1]))
    print('     strongest signals: %s'
          % ', '.join('%s % .3f' % (k, v) for k, v in top.items()))
    return dict(label=label, auc=auc, fpr=fpr, tpr=tpr)


age_train = X_train.index.map(data_df.Age).to_numpy()
age_test = X_test.index.map(data_df.Age).to_numpy()

probes = [
    proxy_recoverability(X_train.Gender_Male.to_numpy(),
                         X_test.Gender_Male.to_numpy(),
                         ['Gender_Male'], 'Gender, from the other 58'),
    proxy_recoverability(X_train.MaritalStatus_Single.to_numpy(),
                         X_test.MaritalStatus_Single.to_numpy(),
                         ['MaritalStatus_Married', 'MaritalStatus_Single'],
                         'Single, from the other 57'),
    proxy_recoverability((age_train >= AGE_CUT).astype(int),
                         (age_test >= AGE_CUT).astype(int),
                         ['Generation_Generation X', 'Generation_Millenials'],
                         '40 and over, Generation dropped'),
    proxy_recoverability((age_train >= AGE_CUT).astype(int),
                         (age_test >= AGE_CUT).astype(int),
                         [], '40 and over, all 59 features'),
]

Gender, from the other 58        AUC 0.4196  from 58 features
     strongest signals: JobRole_Laboratory Technician  0.445, JobRole_Manufacturing Director -0.416, OverTime_Yes -0.234
Single, from the other 57        AUC 0.9505  from 57 features
     strongest signals: StockOptionLevel -4.735, JobRole_Human Resources -0.483, JobSatisfaction_Very High  0.440
40 and over, Generation dropped  AUC 0.6908  from 57 features
     strongest signals: compa_ratio -0.240, First_job_ind -0.202, Education_Below College -0.197
40 and over, all 59 features     AUC 0.9339  from 59 features
     strongest signals: Generation_Millenials -4.525, compa_ratio -0.784, First_job_ind -0.686


In [23]:
traces = [go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='chance',
                     line=dict(color=GRAY, width=1.5, dash='dot'),
                     hoverinfo='skip')]
for colour, probe in zip([BLUE, ORANGE, GREEN, RED], probes):
    traces.append(go.Scatter(
        x=probe['fpr'], y=probe['tpr'], mode='lines',
        name='%s: AUC %.4f' % (probe['label'], probe['auc']),
        line=dict(color=colour, width=2),
        hovertemplate=('<b>' + probe['label'] + '</b>'
                       '<br>false positive rate: %{x:.4f}'
                       '<br>true positive rate: %{y:.4f}<extra></extra>')))

layout = go.Layout(dict(
    title=dict(text='Recovering a protected attribute after deleting it',
               font=dict(size=15)),
    height=560, width=880,
    xaxis=dict(title='false positive rate', range=[0, 1]),
    yaxis=dict(title='true positive rate', range=[0, 1.02]),
    plot_bgcolor='rgba(240, 240, 240, 0.95)',
    paper_bgcolor='rgba(240, 240, 240, 0.95)',
    legend=dict(orientation='h', y=-0.22)))
py.iplot(go.Figure(data=traces, layout=layout))

In [24]:
print(pd.crosstab(data_df.MaritalStatus, data_df.StockOptionLevel,
                  margins=True))
print('\nEvery single employee in this file has StockOptionLevel 0.')

StockOptionLevel    0    1    2   3   All
MaritalStatus                            
Divorced            8  195   75  49   327
Married           153  401   83  36   673
Single            470    0    0   0   470
All               631  596  158  85  1470



Every single employee in this file has StockOptionLevel 0.


Three attributes, three different answers, which is why this is worth ten lines rather than
an opinion.

**Gender: AUC 0.4196.** Worse than chance. The other 58 features carry essentially nothing
about it. Here, and only here, deleting the column really would remove the attribute from the
model. That is a property of how IBM generated this file, and I would not expect it to hold
in a real HR system where job role, department and shift pattern are all correlated with
gender.

**Single: AUC 0.9505.** Delete both marital status columns and the model can still pick out
single employees almost perfectly. The crosstab above says why: in this dataset every single
employee has `StockOptionLevel` 0. A benefits administration field is a near-perfect proxy for
marital status. Nobody designed that; it fell out of the generator, and this is exactly how
proxies appear in real data too.

**40 and over: AUC 0.9339 with all features, 0.6908 once the two `Generation` dummies are
dropped.** The first number is unsurprising, since `Generation` is age with a haircut. The
second is the interesting one: with every age-derived column removed, tenure, income and
job level still recover the age band well above chance. Age is the hardest attribute to
remove from an HR dataset, because almost everything an HR system records accumulates over
time.

So: **do not report "we removed the protected attributes" as a mitigation without running
this test.** It takes ten lines, and on two of these three attributes it would have been
false.

---

# Part 2: what this costs the business

An audit that ends at "the ratio is 0.3867" gets filed. The second half of the job is
turning the confusion matrix into sentences a manager can act on, and then showing that the
decision everybody treats as technical, the 0.5 threshold, is the one with the most money
attached to it.

## 2.1 The confusion matrix in the language of the business

Four numbers, four sentences.

In [25]:
n_test = TN + FP + FN + TP
print('Of %d employees in the held-out set, %d actually left.\n' % (n_test, FN + TP))
for count, event, meaning in [
        (TP, 'the model flagged, and they left.',
         'a retention conversation that had a reason'),
        (FN, 'the model missed, and they left.',
         'a resignation nobody saw coming'),
        (FP, 'the model flagged, and they stayed.',
         'a conversation spent on someone who was not going'),
        (TN, 'the model left alone, and they stayed.',
         'correct, and free')]:
    print('  %3d  %-42s -> %s' % (count, event, meaning))
print('\nSo: it catches %d of %d leavers (%.1f%%), misses %d, and starts %d '
      'conversations\nwith people who were staying anyway. It flags %d people '
      'in total.'
      % (TP, FN + TP, 100 * TP / (FN + TP), FN, FP, TP + FP))

Of 294 employees in the held-out set, 47 actually left.

   27  the model flagged, and they left.          -> a retention conversation that had a reason
   20  the model missed, and they left.           -> a resignation nobody saw coming
   11  the model flagged, and they stayed.        -> a conversation spent on someone who was not going
  236  the model left alone, and they stayed.     -> correct, and free

So: it catches 27 of 47 leavers (57.4%), misses 20, and starts 11 conversations
with people who were staying anyway. It flags 38 people in total.


That is the whole model, in the only terms that matter to the person who has to staff the
follow-up. 38 conversations to book. 27 of them with someone who was genuinely going to
leave. 20 people who will resign with no warning from this system.

Whether that is a good deal depends entirely on two numbers this notebook does not know.

## 2.2 The cost model is a parameter, not a claim

You will find a widely repeated figure for what replacing an employee costs, usually
expressed as a multiple of salary. **I am not going to quote one, because I cannot source
one, and a number with no source in a model that decides who gets a conversation is worse
than no number at all.** It would also be the wrong shape: what it costs your company to
replace a senior engineer is not what it costs to replace a graduate, and neither is what it
costs the company down the road.

So the cost model here is three named parameters that you set, and the arithmetic is done in
front of you:

- `cost_of_replacing`, what the business pays when someone leaves and has to be replaced.
- `cost_of_a_retention_conversation`, what one intervention costs: the manager's time, and
  whatever is offered.
- `conversation_success_rate`, the share of genuine leavers who stay after that conversation.
  Nobody can hand you this either. If it is zero, the model has no value at any threshold,
  and that is the first thing to establish before deploying anything.

The values below are placeholders, chosen so that the arithmetic has numbers in it. Change
them and re-run. A useful property falls out: **only the ratio of the first two matters**,
so you do not have to know what replacing someone costs. You have to know how many
retention conversations it is worth.

In [26]:
cost_of_replacing = 20.0                 # per leaver who leaves and is replaced
cost_of_a_retention_conversation = 1.0   # per person flagged
conversation_success_rate = 0.30         # share of genuine leavers who stay

cost_ratio = cost_of_replacing / cost_of_a_retention_conversation
print('cost ratio: one replacement costs as much as %.0f conversations'
      % cost_ratio)


def expected_cost(tp, fp, fn, replace=None, talk=None, success=None):
    """Total expected cost of running the model at one operating point.

    Every flagged person costs one conversation. Every leaver we missed costs a
    replacement. Every leaver we caught costs a replacement only when the
    conversation fails, which is (1 - success) of the time.
    """
    replace = cost_of_replacing if replace is None else replace
    talk = cost_of_a_retention_conversation if talk is None else talk
    success = conversation_success_rate if success is None else success
    return (tp + fp) * talk + (fn + (1 - success) * tp) * replace


do_nothing = (FN + TP) * cost_of_replacing
at_half = expected_cost(TP, FP, FN)
print('\ndo nothing         : %d leavers x %.0f = %.1f'
      % (FN + TP, cost_of_replacing, do_nothing))
print('model at 0.5       : (%d + %d) x %.0f  +  (%d + %.2f x %d) x %.0f = %.1f'
      % (TP, FP, cost_of_a_retention_conversation, FN,
         1 - conversation_success_rate, TP, cost_of_replacing, at_half))
print('difference         : %.1f, in units of one retention conversation'
      % (do_nothing - at_half))

break_even = (TP + FP) / (conversation_success_rate * TP)
print('\nbreak-even: at 0.5 the model pays for itself once one replacement is '
      'worth\nmore than (%d + %d) / (%.2f x %d) = %.4f conversations.'
      % (TP, FP, conversation_success_rate, TP, break_even))

cost ratio: one replacement costs as much as 20 conversations

do nothing         : 47 leavers x 20 = 940.0
model at 0.5       : (27 + 11) x 1  +  (20 + 0.70 x 27) x 20 = 816.0
difference         : 124.0, in units of one retention conversation

break-even: at 0.5 the model pays for itself once one replacement is worth
more than (27 + 11) / (0.30 x 27) = 4.6914 conversations.


The break-even line is the sentence to take into the meeting. At this threshold, with a 30%
success rate, the model is worth running as soon as replacing one employee costs more than
about 4.7 retention conversations. That is a question a finance partner can answer in an
afternoon, and it does not require anybody to accept an industry statistic.

It also inverts cleanly. If you believe your success rate is 10% rather than 30%, the
break-even moves to 14.07 conversations per replacement, because the same 38 conversations
now save a third as many people. Try it in the cell above.

## 2.3 The 0.5 threshold is a business decision

Nothing in scikit-learn chose 0.5 for a reason. `predict()` picks it because it has to pick
something. Every operating point on the curve below is available at no cost, and they differ
enormously in what they do to the business.

Sweep it, and put catches, false alarms and expected cost on the same axis.

In [27]:
thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
sweep = pd.DataFrame({'threshold': thresholds})
sweep['catches'] = [int(((y_true == 1) & (y_score >= t)).sum())
                    for t in thresholds]
sweep['false_alarms'] = [int(((y_true == 0) & (y_score >= t)).sum())
                         for t in thresholds]
sweep['missed'] = (FN + TP) - sweep.catches
sweep['flagged'] = sweep.catches + sweep.false_alarms
sweep['expected_cost'] = expected_cost(sweep.catches, sweep.false_alarms,
                                       sweep.missed)

best = sweep.loc[sweep.expected_cost.idxmin()]
half = sweep.loc[sweep.threshold == 0.50].iloc[0]
print('cheapest threshold at a cost ratio of %.0f and a %.0f%% success rate:'
      % (cost_ratio, conversation_success_rate * 100))
print('  threshold %.2f: %d catches, %d false alarms, expected cost %.1f'
      % (best.threshold, best.catches, best.false_alarms, best.expected_cost))
print('  threshold 0.50: %d catches, %d false alarms, expected cost %.1f'
      % (half.catches, half.false_alarms, half.expected_cost))
print('  doing nothing :  0 catches,  0 false alarms, expected cost %.1f'
      % do_nothing)
print('\nMoving off the default saves %.1f conversations worth of cost, and '
      'catches %d\nmore of the %d leavers, at the price of %d more false '
      'alarms.'
      % (half.expected_cost - best.expected_cost,
         best.catches - half.catches, FN + TP,
         best.false_alarms - half.false_alarms))

cheapest threshold at a cost ratio of 20 and a 30% success rate:
  threshold 0.23: 37 catches, 34 false alarms, expected cost 789.0
  threshold 0.50: 27 catches, 11 false alarms, expected cost 816.0
  doing nothing :  0 catches,  0 false alarms, expected cost 940.0

Moving off the default saves 27.0 conversations worth of cost, and catches 10
more of the 47 leavers, at the price of 23 more false alarms.


In [28]:
detail = np.stack([sweep.catches, sweep.false_alarms, sweep.expected_cost,
                   sweep.missed], axis=-1)
hover = ('threshold %{x:.2f}<br>catches: %{customdata[0]} of '
         + str(FN + TP) + ' leavers<br>false alarms: %{customdata[1]}'
         '<br>missed: %{customdata[3]}'
         '<br>expected cost: %{customdata[2]:.1f} conversations<extra></extra>')

traces = [
    go.Scatter(x=sweep.threshold, y=sweep.catches, name='catches',
               line=dict(color=GREEN, width=2.4), customdata=detail,
               hovertemplate=hover),
    go.Scatter(x=sweep.threshold, y=sweep.false_alarms, name='false alarms',
               line=dict(color=ORANGE, width=2.4), customdata=detail,
               hovertemplate=hover),
    go.Scatter(x=sweep.threshold, y=sweep.expected_cost, name='expected cost',
               yaxis='y2', line=dict(color=BLUE, width=2.4, dash='dash'),
               customdata=detail, hovertemplate=hover),
    go.Scatter(x=[best.threshold], y=[best.expected_cost], yaxis='y2',
               name='cheapest at ratio %.0f' % cost_ratio, mode='markers',
               marker=dict(color=RED, size=13, symbol='diamond',
                           line=dict(color=INK, width=1)),
               hovertemplate=('cheapest: threshold %{x:.2f}'
                              '<br>expected cost %{y:.1f}<extra></extra>')),
    go.Scatter(x=[0.5, 0.5], y=[0, sweep.false_alarms.max()], mode='lines',
               name='scikit-learn default', line=dict(color=GRAY, width=1.5,
                                                      dash='dot'),
               hoverinfo='skip'),
]
layout = go.Layout(dict(
    title=dict(text='Every threshold is available. Drag along the curve.',
               font=dict(size=15)),
    height=560, width=900,
    xaxis=dict(title='threshold on the predicted score'),
    yaxis=dict(title='people'),
    yaxis2=dict(title='expected cost, in retention conversations',
                overlaying='y', side='right', showgrid=False),
    plot_bgcolor='rgba(240, 240, 240, 0.95)',
    paper_bgcolor='rgba(240, 240, 240, 0.95)',
    legend=dict(orientation='h', y=-0.22)))
py.iplot(go.Figure(data=traces, layout=layout))

Drag along the blue curve and the tradeoff becomes obvious in a way no table makes it. At a
cost ratio of 20 to 1 the cheapest place to stand is 0.23, not 0.5: ten more leavers reached,
23 more conversations that turn out to be unnecessary, and a lower expected bill.

The shape is worth reading too. The cost curve is flat over a wide middle. That is good news
for anyone who has to defend a threshold, because it means the exact choice matters much less
than being roughly in the right region, and being in the right region requires only a rough
sense of the cost ratio.

But the cheapest threshold does move with the parameters, and it is worth seeing how much
before anybody quotes 0.23 as a recommendation.

In [29]:
ratio_grid = [5, 10, 20, 40, 80]
success_grid = [0.10, 0.20, 0.30, 0.40, 0.50]

optimal = np.zeros((len(ratio_grid), len(success_grid)))
for i, ratio_value in enumerate(ratio_grid):
    for j, success_value in enumerate(success_grid):
        costs = expected_cost(sweep.catches, sweep.false_alarms, sweep.missed,
                              replace=ratio_value, talk=1.0,
                              success=success_value)
        optimal[i, j] = sweep.threshold[int(np.argmin(costs.to_numpy()))]

figure = ff.create_annotated_heatmap(
    z=optimal, x=['%.0f%%' % (s * 100) for s in success_grid],
    y=['%d to 1' % r for r in ratio_grid],
    annotation_text=[['%.2f' % v for v in row] for row in optimal],
    colorscale='Viridis', reversescale=True, showscale=True,
    hovertemplate=('conversation success %{x}<br>cost ratio %{y}'
                   '<br>cheapest threshold: %{z:.2f}<extra></extra>'))
figure.update_layout(dict(
    title=dict(text='Cheapest threshold, over the two numbers you have to '
                    'supply', font=dict(size=15)),
    height=470, width=760,
    xaxis=dict(title='conversation success rate', side='bottom'),
    yaxis=dict(title='replacement cost, in conversations'),
    plot_bgcolor='rgba(240, 240, 240, 0.95)',
    paper_bgcolor='rgba(240, 240, 240, 0.95)'))
py.iplot(figure)

The cheapest threshold spans almost the whole range, from 0.97 where replacement is cheap and
the conversation rarely works (so flag almost nobody) to 0.01 where replacement is expensive
and the conversation usually works (so talk to everybody). The choice of operating point is
not a detail to be settled by a default; it is the largest lever in the deployment, and it is
set by two numbers that live in the business rather than in the model.

That is the single most useful thing to take away from this notebook: **if you have not
chosen your threshold on purpose, someone else's default is making your budget decisions.**

## 2.4 What the fairness gap costs, in the same units

Now put the two halves together, because this is the part that usually goes missing. The
audit found one gap that survives its interval: the model reaches 24 of the 35 leavers under
40, and 3 of the 12 leavers aged 40 and over.

Express that as reach, and then as money.

In [30]:
print('retention conversations that actually reached a leaver:')
for label in ['Under 40', '40 and over']:
    r = audit[(audit.dimension == 'AgeBand') & (audit.group == label)].iloc[0]
    saved = conversation_success_rate * r.tp
    print('  %-12s %2d of %2d leavers reached (%.4f), %2d missed. '
          'Expected retentions: %.2f'
          % (label, r.tp, r.leavers, r.tpr, r.fn, saved))

older = audit[(audit.dimension == 'AgeBand') &
              (audit.group == '40 and over')].iloc[0]
younger = audit[(audit.dimension == 'AgeBand') &
                (audit.group == 'Under 40')].iloc[0]

# What the older band would have received at the younger band's catch rate.
at_parity = younger.tpr * older.leavers
shortfall = at_parity - older.tp
print('\nAt the younger band\'s catch rate, %.2f of the %d older leavers would '
      'have been\nreached instead of %d. The shortfall is %.2f conversations, '
      'worth %.2f expected\nretentions, worth %.1f in the cost units above.'
      % (at_parity, older.leavers, older.tp, shortfall,
         shortfall * conversation_success_rate,
         shortfall * conversation_success_rate * cost_of_replacing))

retention conversations that actually reached a leaver:
  Under 40     24 of 35 leavers reached (0.6857), 11 missed. Expected retentions: 7.20
  40 and over   3 of 12 leavers reached (0.2500),  9 missed. Expected retentions: 0.90

At the younger band's catch rate, 8.23 of the 12 older leavers would have been
reached instead of 3. The shortfall is 5.23 conversations, worth 1.57 expected
retentions, worth 31.4 in the cost units above.


Roughly five conversations, one and a half expected retentions, about thirty conversations
worth of cost on this test set of 294 people. Small numbers, because it is a small test set
of imaginary people. The point is the shape of the argument, and the shape survives scaling.

Say it in both registers, because both are true and each one persuades a different person in
the room:

- **The equity failure.** One group systematically receives less of the intervention. Nobody
  in that group is told they were scored, ranked and skipped. They simply resign, and the
  company records it as a surprise.
- **The business failure.** You are paying to retain a segment and then under-serving it. The
  employees you fail to reach are not cheaper to replace than the ones you do reach; if
  anything, on this dataset, they are older, longer-tenured and more expensive. The gap is
  not a tax you pay for accuracy. It is accuracy you did not get.

That second framing is the one that moves budget, and it is why a fairness audit belongs in
the same document as the cost model rather than in an appendix behind it.

## 2.5 What a reviewer should ask before this model is deployed

Short and concrete. If you are the person signing off, these are the questions, and "we ran
the notebook" is not an answer to any of them.

1. **What is the intervention, exactly?** A conversation, a raise, a transfer, or a note in a
   file that follows the employee? The entire audit changes depending on whether being
   flagged is a benefit or a mark.
2. **Who sees the score?** The employee's own manager, HR, or the promotion committee? A
   score built to trigger retention and then read at performance review is a different system
   from the one that was audited.
3. **What is the threshold, and who chose it?** If the answer is 0.5, nobody chose it. Ask
   for the cost ratio and the assumed success rate that justify it, in writing.
4. **Show me the true positive rate by group, with intervals.** Not the accuracy. Not the
   AUC. The share of each group's leavers the model reaches, and the interval on each one.
5. **Which groups are too small to audit?** Name them. A group the audit cannot see is not a
   group the audit cleared. Here, Human Resources has 11 people and 1 leaver.
6. **Was the protected attribute really removed, or just the column?** Ask for the proxy AUC.
   Ten lines of code, and on this dataset it would have contradicted the claim on two
   attributes out of three.
7. **What happens to someone who is flagged and then leaves anyway?** If nothing, the model
   is generating work with no feedback loop, and the success rate in the cost model is
   unmeasurable forever.
8. **When is it re-audited, and against what?** A model whose base rates shift, which is
   every HR model, drifts out of whatever fairness position it was signed off in.
9. **Who is accountable for the number this model moves?** If no one owns retention, the
   model is decoration and the audit is decoration about decoration.

---

## The one-page summary

Everything above, on a single image, for the reader who will never open a notebook. It is
built with matplotlib rather than plotly because it has to leave the browser: a slide, a
post, a page in a deck.

The numbers on it come from the `HEADLINE` dictionary below, which is assembled from the
same audit table as everything else and printed before it is drawn. Nothing on the image is
typed in by hand. That is the only way to keep a summary graphic honest as the analysis
underneath it changes.

In [31]:
older_ci = wilson(older.tp, older.leavers)
younger_ci = wilson(younger.tp, younger.leavers)
gender_ratio, gender_low, gender_high = parity_ratio(audit, 'Gender',
                                                     'selection_rate')
age_ratio, age_low, age_high = parity_ratio(audit, 'AgeBand', 'selection_rate')
marital_ratio, _, _ = parity_ratio(audit, 'MaritalStatus', 'selection_rate')

HEADLINE = {
    'test_n': int(n_test), 'test_leavers': int(FN + TP),
    'tn': TN, 'fp': FP, 'fn': FN, 'tp': TP,
    'tpr_young': float(younger.tpr), 'tpr_old': float(older.tpr),
    'tp_young': int(younger.tp), 'leavers_young': int(younger.leavers),
    'tp_old': int(older.tp), 'leavers_old': int(older.leavers),
    'young_ci': [float(v) for v in younger_ci],
    'old_ci': [float(v) for v in older_ci],
    'gap': float(gaps['AgeBand']['gap']),
    'gap_ci': [float(gaps['AgeBand']['lo']), float(gaps['AgeBand']['hi'])],
    'gender_gap': float(gaps['Gender']['gap']),
    'gender_gap_ci': [float(gaps['Gender']['lo']),
                      float(gaps['Gender']['hi'])],
    'dp_gender': float(gender_ratio), 'dp_marital': float(marital_ratio),
    'dp_age': float(age_ratio), 'four_fifths': FOUR_FIFTHS,
    'best_threshold': float(best.threshold),
    'best_catches': int(best.catches),
    'best_false_alarms': int(best.false_alarms),
    'half_catches': int(half.catches),
    'half_false_alarms': int(half.false_alarms),
    'cost_ratio': float(cost_ratio),
    'success': float(conversation_success_rate),
    'proxy_single': float(probes[1]['auc']),
    'proxy_age': float(probes[3]['auc']),
    'proxy_gender': float(probes[0]['auc']),
}
for key, value in HEADLINE.items():
    print('%-20s %s' % (key, value))

test_n               294
test_leavers         47
tn                   236
fp                   11
fn                   20
tp                   27
tpr_young            0.6857142857142857
tpr_old              0.25
tp_young             24
leavers_young        35
tp_old               3
leavers_old          12
young_ci             [0.5202023638793081, 0.8144915532573418]
old_ci               [0.08894166839405471, 0.5323053349335657]
gap                  0.4357142857142857
gap_ci               [0.10846743943236897, 0.6419262417584164]
gender_gap           0.10606060606060608
gender_gap_ci        [-0.18110695256496057, 0.38077601949487755]
dp_gender            0.6061598951507209
dp_marital           0.3333333333333333
dp_age               0.38666666666666666
four_fifths          0.8
best_threshold       0.23
best_catches         37
best_false_alarms    34
half_catches         27
half_false_alarms    11
cost_ratio           20.0
success              0.3
proxy_single         0.9504901960784314


In [32]:
import textwrap

# A new filename. Nothing already in img/ is touched: the figures the main
# notebook and the README depend on are not this notebook's to regenerate.
OUT = IMG / 'fairness_audit_summary.png'

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left', 'axes.titlecolor': INK,
    'axes.labelsize': 9.5, 'axes.labelcolor': '#444444',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#999999', 'xtick.color': '#666666',
    'ytick.color': '#666666', 'legend.frameon': False,
})

fig = plt.figure(figsize=(13.33, 8.4), dpi=150, constrained_layout=True)
fig.patch.set_facecolor('#fbfaf7')
# Every element lives inside a gridspec cell. No absolute placement anywhere:
# change a font and the layout re-solves instead of colliding.
grid = fig.add_gridspec(3, 2, height_ratios=[0.40, 1.0, 1.0],
                        width_ratios=[1.0, 1.0])

# --- header -----------------------------------------------------------------
head = fig.add_subplot(grid[0, :])
head.axis('off')
head.text(0, 0.86, 'Auditing an attrition model: what it would mean deployed',
          fontsize=21, fontweight='bold', color=INK, va='top')
head.text(0, 0.40,
          'One logistic regression, %d held-out employees, %d of whom left. '
          'Every rate below is computed from that confusion matrix.'
          % (HEADLINE['test_n'], HEADLINE['test_leavers']),
          fontsize=11.5, color='#555555', va='top')
head.text(0, 0.06,
          'IBM sample data. The employees are fictional, so this measures the '
          'method, not any real workforce.',
          fontsize=10.5, color=RED, va='top', style='italic')

# --- panel A: equal opportunity by age band ---------------------------------
axis_a = fig.add_subplot(grid[1, 0])
bands = ['Under 40', '40 and over']
rates = [HEADLINE['tpr_young'], HEADLINE['tpr_old']]
lows = [HEADLINE['young_ci'][0], HEADLINE['old_ci'][0]]
highs = [HEADLINE['young_ci'][1], HEADLINE['old_ci'][1]]
positions = np.arange(len(bands))
axis_a.barh(positions, rates, height=0.42, color=[GREEN, RED],
            edgecolor=INK, linewidth=0.7, zorder=2)
axis_a.errorbar(rates, positions,
                xerr=[np.array(rates) - np.array(lows),
                      np.array(highs) - np.array(rates)],
                fmt='none', ecolor='#555555', elinewidth=1.5, capsize=5,
                zorder=3)
for position, rate, caught, total in zip(
        positions, rates, [HEADLINE['tp_young'], HEADLINE['tp_old']],
        [HEADLINE['leavers_young'], HEADLINE['leavers_old']]):
    axis_a.text(rate + 0.02, position - 0.30,
                '%.0f%%   (%d of %d leavers reached)'
                % (rate * 100, caught, total), fontsize=10, color=INK,
                va='center')
axis_a.set_yticks(positions, bands, fontsize=11)
axis_a.set_xlim(0, 1.0)
axis_a.set_ylim(1.85, -0.55)
axis_a.set_xlabel('share of the group\'s leavers the model catches')
axis_a.set_title('The one gap that survives its own interval')
axis_a.grid(axis='x', alpha=0.25, zorder=0)
axis_a.text(0.01, 0.05,
            textwrap.fill(
                'gap %.2f, 95%% interval [%.2f, %.2f], excludes zero. The same '
                'test on gender gives [%.2f, %.2f], which does not.'
                % (HEADLINE['gap'], HEADLINE['gap_ci'][0],
                   HEADLINE['gap_ci'][1], HEADLINE['gender_gap_ci'][0],
                   HEADLINE['gender_gap_ci'][1]), 62),
            fontsize=9.2, color='#555555', va='bottom',
            transform=axis_a.transAxes)

# --- panel B: four-fifths -----------------------------------------------------
axis_b = fig.add_subplot(grid[1, 1])
names = ['Gender\nF vs M', 'Marital status\nDivorced vs Single',
         'Age band\n40+ vs under 40']
ratios = [HEADLINE['dp_gender'], HEADLINE['dp_marital'], HEADLINE['dp_age']]
axis_b.bar(np.arange(3), ratios, width=0.5, color=ORANGE, edgecolor=INK,
           linewidth=0.7, zorder=2)
axis_b.axhline(HEADLINE['four_fifths'], color=RED, linewidth=1.8,
               linestyle='--', zorder=3)
axis_b.text(2.46, HEADLINE['four_fifths'] + 0.02, 'four-fifths rule',
            color=RED, fontsize=9.5, ha='right')
for index, value in enumerate(ratios):
    axis_b.text(index, value + 0.03, '%.2f' % value, ha='center', fontsize=11,
                color=INK, fontweight='bold')
axis_b.set_xticks(np.arange(3), names, fontsize=9.5)
axis_b.set_xlim(-0.6, 2.6)
axis_b.set_ylim(0, 1.05)
axis_b.set_ylabel('selection rate ratio, worst over best')
axis_b.set_title('Who gets flagged: three attributes, three failures')
axis_b.grid(axis='y', alpha=0.25, zorder=0)

# --- panel C: the business tradeoff ------------------------------------------
axis_c = fig.add_subplot(grid[2, 0])
axis_c.plot(sweep.threshold, sweep.expected_cost, color=BLUE, linewidth=2.2,
            zorder=2)
axis_c.axvline(0.5, color=GRAY, linewidth=1.4, linestyle=':', zorder=1)
axis_c.plot([HEADLINE['best_threshold']],
            [sweep.expected_cost[sweep.threshold ==
                                 HEADLINE['best_threshold']].iloc[0]],
            marker='D', color=RED, markersize=9, markeredgecolor=INK, zorder=3)
axis_c.annotate('cheapest at %.2f\n%d caught, %d false alarms'
                % (HEADLINE['best_threshold'], HEADLINE['best_catches'],
                   HEADLINE['best_false_alarms']),
                xy=(HEADLINE['best_threshold'],
                    sweep.expected_cost[sweep.threshold ==
                                        HEADLINE['best_threshold']].iloc[0]),
                xytext=(0.38, 0.72), textcoords='axes fraction', fontsize=9.5,
                color=INK,
                arrowprops=dict(arrowstyle='->', color='#888888'))
axis_c.annotate('scikit-learn default 0.5\n%d caught, %d false alarms'
                % (HEADLINE['half_catches'], HEADLINE['half_false_alarms']),
                xy=(0.5, 0.16), xycoords=('data', 'axes fraction'),
                xytext=(0.62, 0.30), textcoords='axes fraction', fontsize=9.5,
                color='#555555',
                arrowprops=dict(arrowstyle='->', color='#888888'))
axis_c.set_xlabel('threshold on the predicted score')
axis_c.set_ylabel('expected cost, in conversations')
axis_c.set_title('0.5 is a business decision, not a default')
axis_c.grid(alpha=0.25, zorder=0)

# --- panel D: what to do about it --------------------------------------------
axis_d = fig.add_subplot(grid[2, 1])
axis_d.axis('off')
axis_d.set_title('What the audit changes')
lines = [
    ('Reach, not accuracy.',
     '%d of %d older leavers reached against %d of %d younger. Accuracy hides '
     'this entirely.' % (HEADLINE['tp_old'], HEADLINE['leavers_old'],
                         HEADLINE['tp_young'], HEADLINE['leavers_young'])),
    ('Deleting the column is not a fix.',
     'Marital status is recoverable from the rest at AUC %.2f, the age band at '
     '%.2f. Gender, here, at %.2f.'
     % (HEADLINE['proxy_single'], HEADLINE['proxy_age'],
        HEADLINE['proxy_gender'])),
    ('Pick the threshold on purpose.',
     'At %.0f conversations per replacement and a %.0f%% success rate, %.2f '
     'beats 0.5 on cost and on reach.'
     % (HEADLINE['cost_ratio'], HEADLINE['success'] * 100,
        HEADLINE['best_threshold'])),
    ('Size every group first.',
     'Human Resources holds 11 people and 1 leaver. A group the audit cannot '
     'see is not a group it cleared.'),
]
y_position = 0.96
for heading, body in lines:
    axis_d.text(0.0, y_position, heading, fontsize=11, fontweight='bold',
                color=INK, va='top', transform=axis_d.transAxes)
    axis_d.text(0.0, y_position - 0.075, textwrap.fill(body, 66),
                fontsize=9.6, color='#555555', va='top',
                transform=axis_d.transAxes)
    y_position -= 0.25

fig.savefig(OUT, facecolor=fig.get_facecolor())
plt.close(fig)
print('wrote %s (%.0f KB)' % (OUT.name, OUT.stat().st_size / 1024))

wrote fairness_audit_summary.png (272 KB)


<img src="../img/fairness_audit_summary.png" alt="Fairness audit summary" style="width: 100%;"/>

## Takeaways

1. **Every metric here is four integers and a division.** Selection rate, true positive rate,
   false positive rate, predictive value: all of them come out of a confusion matrix you can
   slice with a boolean mask. The libraries are convenience, not capability, and writing the
   arithmetic once is how you learn which metric answers which question.
2. **Pick the metric from the decision, not from a list.** Here the decision is who gets a
   retention conversation, so the metric that matters is the share of each group's leavers the
   model reaches. Demographic parity fails on all four dimensions audited and most of that
   failure is base rates, which is why it is a screening question rather than a verdict.
3. **Put an interval on everything, and put it on the gap rather than on each rate.** Two of
   the three gaps in this audit dissolve once you do. Comparing error bars by eye would have
   thrown away the one that does not.
4. **Test the proxy claim instead of asserting it.** Ten lines. On this data, deleting the
   marital status columns leaves the attribute recoverable at AUC 0.9505, because every single
   employee has the same stock option level.
5. **The threshold is the biggest lever in the deployment and it is usually the least
   discussed.** The cheapest operating point moved from 0.97 to 0.01 across a plausible range
   of two business parameters. Nobody should be shipping a `predict()` call.
6. **A fairness gap and a business loss are the same sentence read twice.** The group the
   model reaches least is a group you are paying to keep and failing to serve.

## What this notebook does not do

Stated plainly, because an audit that hides its own limits is not an audit.

- **The people are fictional.** Third time. It measures the method.
- **One test set, 294 people, drawn once at `random_state=1234`.** Every number here is
  conditional on that draw. Repeated splits or a bootstrap would give a better sense of the
  spread than the intervals here, which only account for sampling within this one split.
- **Only three attributes, taken one at a time.** No intersection: this notebook never asks
  about women over 40, because the test set holds too few of them to support the question.
  Real audits find their worst results at intersections, and this dataset cannot go there.
- **No mitigation.** Reweighting, threshold-per-group, and constrained training are all real
  options with real costs, and setting a different threshold per age band raises a legal
  question in most jurisdictions that a notebook cannot answer. Measuring first is the point
  here.
- **The cost model is a parameter, not a finding.** Every currency figure above is in units of
  one retention conversation, and the ratio and success rate are placeholders. If you carry
  any number from Part 2 into a meeting without replacing them, you have carried my arithmetic
  and not your business.

---

*Satsawat Natakarnkitkul . [satsawat.ai](https://satsawat.ai) .
[AI in Practice newsletter](https://satsawat.ai/#newsletter)*